
# Transform Orders Data - Explode Arrays

1. Access elements from the JSON Object
2. Deduplicate Array Elements
3. Explode Arrays
4. Write the Transformed Data to Silver Schema

In [0]:
SELECT * FROM 
gizmobox.silver.orders_json


## 1. Access elements from the JSON Object

In [0]:
SELECT json_value.order_id, 
      json_value.order_status, 
      json_value.payment_method, 
      json_value.total_amount,
      json_value.transaction_timestamp, 
      json_value.customer_id,
      json_value.items
FROM gizmobox.silver.orders_json



## 2. Deduplicate Array Elements

In [0]:
SELECT json_value.order_id, 
      json_value.order_status, 
      json_value.payment_method, 
      json_value.total_amount,
      json_value.transaction_timestamp, 
      json_value.customer_id,
      array_distinct(json_value.items) as items
FROM gizmobox.silver.orders_json


## 3. Explode Arrays

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tmp_orders 
AS
SELECT json_value.order_id, 
      json_value.order_status, 
      json_value.payment_method, 
      json_value.total_amount,
      json_value.transaction_timestamp, 
      json_value.customer_id,
      explode(array_distinct(json_value.items)) as item
FROM gizmobox.silver.orders_json

In [0]:
SELECT * FROM tmp_orders


## 4. Write the Transformed Data to silver Schema


In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders
AS
SELECT order_id, 
        order_status,
        payment_method, 
        total_amount, 
        transaction_timestamp, 
        customer_id,
        item.category, 
        item.details.brand, 
        item.details.color, 
        item.item_id, 
        item.name, 
        item.price, 
        item.quantity
FROM tmp_orders

In [0]:
SELECT * FROM gizmobox.silver.orders;